# Clustering y búsqueda semántica de reviews
Pipeline limpio y comentado: genera embeddings con SentenceTransformers, reduce dimensión con UMAP, agrupa con HDBSCAN, etiqueta automáticamente cada cluster, construye un índice FAISS para búsqueda semántica y produce un muestreo estratificado de 20 reviews listo para CSV. Usa las mismas rutas/artefactos que el cuaderno anterior y agrega contexto paso a paso.


## Guía rápida de la estructura
- Configuración y carga de datos.
- Funciones reutilizables (encoding, reducción, clustering, etiquetado, búsqueda).
- Generación de embeddings y reducción de dimensión.
- Clustering + catálogo de clusters con etiquetas legibles.
- Índice FAISS y ejemplo de consulta.
- Búsqueda two-stage por centroides de cluster con fallback a ruido.
- Búsqueda rápida: cluster único + FAISS dentro del cluster.
- Muestreo estratificado de 20 reviews (solo id + review) a CSV.

**Notas**
- `cluster_id = -1` es ruido/noise en HDBSCAN.
- Usa el mismo modelo `all-MiniLM-L6-v2`; ajusta rutas/hiperparámetros según necesidad.
- Los nombres de cluster se generan automáticamente con TF-IDF sobre las reviews de cada grupo.


## 0. Configuración e imports
Importamos todas las dependencias en un solo lugar y definimos rutas fijas para que el resto del flujo sea reproducible.


In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import umap
import hdbscan
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer


warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_colwidth", 220)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DATA_PATH = "df_reviews_articles_with_synth_reviews.csv"
EMB_PATH = "review_embeddings.npy"
UMAP_PATH = "umap_model.pk1"
EMB_UMAP_PATH = "embeddings_umap.npy"
CLUSTER_MODEL_PATH = "cluster_model.pk1"
FAISS_INDEX_PATH = "faiss_reviews.index"
DF_EMB_PATH = "df_with_embeddings.pkl"
DF_CLUSTER_PATH = "df_with_clusters.pk1"
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


## 1. Carga y preprocesamiento del dataset
Leemos el CSV de reviews. El texto se fuerza a `str` y se permite activar eliminación de duplicados por contenido si quieres reducir tamaño antes de calcular embeddings.


In [52]:
DROP_DUPLICATES = False  # pon True si quieres eliminar textos repetidos antes de calcular embeddings

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df["review"] = df["review"].fillna("").astype(str)

if DROP_DUPLICATES:
    before = len(df)
    df = df.drop_duplicates(subset="review").reset_index(drop=True)
    print(f"Eliminadas {before - len(df)} filas duplicadas por texto de review.")

print(f"Total filas: {len(df)}")
df.head()


Total filas: 59458


,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag,product_code,...,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,pop_rank,review_stars,review
0,2019-02-25,,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,3,"It's an okay top, the straps are a bit thin but it fits well and is comfortable enough for everyday wear."
1,2018-12-23,fbbc4b14371dac97483160a35dbd93e7a57e292aedbe2cef70940ae205ae887c,108775015,0.007186,2,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4,"I like this top, it's simple and comfy, the black color is perfect for layering or wearing on its own, just wish it came in more sizes."
2,2019-02-03,4c53008e64aa6c59bf1a2b7025ae4afb98be8bc8b457c00e7cae083027adbf63,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4,"Great value for the price, this strap top is soft and comfortable, the narrow straps are a nice touch, and it pairs well with my favorite jeans."
3,2018-12-12,6bdaa2c45d8f21f24bc42f62b873897dec4e5cc00a69cf8d07862645626a6b79,108775015,0.008458,1,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5,"I'm obsessed with this strap top, it's so versatile and comfortable, the quality is top-notch considering the price, and it's perfect for hot summer days."
4,2018-12-03,1033afaf7b151c626d26baa254b851a2d15368b43b215e9f368d97c2f67a045a,108775015,0.008034,1,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5,"This strap top is a staple in my wardrobe, it's incredibly soft, fits perfectly, and the black color is timeless, I highly recommend it."


In [53]:
# cuántos tipos diferentes hay (y opcionalmente listarlos)
unique_types = df_raw["product_type_name"].dropna().unique()
len(unique_types), unique_types

(108,
 array(['Vest top', 'Bra', 'Underwear Tights', 'Leggings/Tights',
        'Trousers', 'Hair clip', 'Umbrella', 'Sweater', 'Socks', 'Unknown',
        'Hoodie', 'Hair/alice band', 'Belt', 'Boots', 'Bikini top',
        'Hair string', 'Swimsuit', 'Skirt', 'Kids Underwear top',
        'T-shirt', 'Pyjama set', 'Dress', 'Sunglasses', 'Gloves',
        'Hat/beanie', 'Cap/peaked', 'Earring', 'Top', 'Blazer',
        'Pyjama jumpsuit/playsuit', 'Swimwear bottom', 'Cardigan',
        'Underwear bottom', 'Jacket', 'Shirt', 'Costumes', 'Robe',
        'Shorts', 'Bodysuit', 'Scarf', 'Coat', 'Other accessories',
        'Polo shirt', 'Slippers', 'Night gown', 'Alice band', 'Straw hat',
        'Tailored Waistcoat', 'Ballerinas', 'Tie', 'Necklace',
        'Pyjama bottom', 'Felt hat', 'Bag', 'Bracelet', 'Watch',
        'Dungarees', 'Swimwear set', 'Underwear body', 'Hat/brim',
        'Flat shoe', 'Jumpsuit/Playsuit', 'Sneakers', 'Sandals', 'Blouse',
        'Wedge', 'Long John', 'Sleeping s

## 2. Funciones utilitarias (encoder, reducción, clustering)
Centralizamos las funciones para que cada paso quede empaquetado:
- `encode_reviews`: usa `SentenceTransformer` en batches.
- `reduce_embeddings`: UMAP para pasar de 384 dims a algo manejable para clustering.
- `cluster_embeddings`: HDBSCAN que encuentra clusters de densidad y marca ruido (-1).


In [54]:
def load_encoder(model_name: str = MODEL_NAME) -> SentenceTransformer:
    # Carga el modelo de sentence-transformers.
    return SentenceTransformer(model_name)


def encode_reviews(texts: pd.Series, model: SentenceTransformer | None = None, batch_size: int = 256, show_progress: bool = True) -> np.ndarray:
    model = model or load_encoder()
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size].tolist()
        batch_emb = model.encode(batch, show_progress_bar=show_progress)
        embeddings.append(batch_emb)
    return np.vstack(embeddings)


def reduce_embeddings(x: np.ndarray, n_neighbors: int = 30, n_components: int = 15, metric: str = "cosine", random_state: int = RANDOM_STATE):
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric=metric,
        random_state=random_state,
    )
    return reducer, reducer.fit_transform(x)


def cluster_embeddings(x: np.ndarray, min_cluster_size: int = 50, metric: str = "euclidean"):
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric=metric,
        cluster_selection_method="leaf",
    )
    return clusterer, clusterer.fit_predict(x)


## 3. Embeddings de las reviews
Calculamos (o cargamos) los embeddings. Si el `.npy` ya existe, se reutiliza para no recalcular. Se guardan también en el DataFrame para exportar a pickle.


In [55]:
reviews = df["review"]
model = load_encoder(MODEL_NAME)

if Path(EMB_PATH).exists():
    embeddings = np.load(EMB_PATH)
    print(f"Cargados embeddings desde {EMB_PATH} -> {embeddings.shape}")
else:
    embeddings = encode_reviews(reviews, model=model, batch_size=256)
    np.save(EMB_PATH, embeddings)
    print(f"Calculados y guardados embeddings en {EMB_PATH} -> {embeddings.shape}")

df["embedding"] = embeddings.tolist()
df.to_pickle(DF_EMB_PATH)
df.head()


Cargados embeddings desde review_embeddings.npy -> (59458, 384)


,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag,product_code,...,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,pop_rank,review_stars,review,embedding
0,2019-02-25,,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,3,"It's an okay top, the straps are a bit thin but it fits well and is comfortable enough for everyday wear.","[0.002053986769169569, 0.020039470866322517, -0.007594653405249119, -0.019627200439572334, 0.06257335096597672, 0.00975802168250084, 0.08956824988126755, 0.08612838387489319, -0.09115290641784668, 0.0457545705139637,..."
1,2018-12-23,fbbc4b14371dac97483160a35dbd93e7a57e292aedbe2cef70940ae205ae887c,108775015,0.007186,2,10841,0.000341,0.03,True,108775,...,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4,"I like this top, it's simple and comfy, the black color is perfect for layering or wearing on its own, just wish it came in more sizes.","[-0.052451241761446, 0.04816248640418053, -0.00997347105294466, 0.014130779542028904, 0.07150337100028992, 0.018838291987776756, 0.04855689778923988, 0.05173642933368683, -0.00796680711209774, 0.055966202169656754, -..."
2,2019-02-03,4c53008e64aa6c59bf1a2b7025ae4afb98be8bc8b457c00e7cae083027adbf63,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4,"Great value for the price, this strap top is soft and comfortable, the narrow straps are a nice touch, and it pairs well with my favorite jeans.","[-0.03485940396785736, 0.02411106415092945, -0.02471686527132988, 0.030055206269025803, 0.03784309700131416, -0.0016616061329841614, 0.14023540914058685, 0.0757829025387764, -0.016435954719781876, 0.02874213084578514..."
3,2018-12-12,6bdaa2c45d8f21f24bc42f62b873897dec4e5cc00a69cf8d07862645626a6b79,108775015,0.008458,1,10841,0.000341,0.03,True,108775,...,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5,"I'm obsessed with this strap top, it's so versatile and comfortable, the quality is top-notch considering the price, and it's perfect for hot summer days.","[-0.055822160094976425, -0.011693540960550308, 0.00857668649405241, 0.06894786655902863, 0.04447505250573158, -0.040184859186410904, 0.09932538866996765, 0.010416699573397636, -0.010817932896316051, 0.031315684318542..."
4,2018-12-03,1033afaf7b151c626d26baa254b851a2d15368b43b215e9f368d97c2f67a045a,108775015,0.008034,1,10841,0.000341,0.03,True,108775,...,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5,"This strap top is a staple in my wardrobe, it's incredibly soft, fits perfectly, and the black color is timeless, I highly recommend it.","[-0.061714280396699905, 0.0028553439769893885, -0.024476978927850723, 0.03282712772488594, 0.0796651691198349, -0.00243640155531466, 0.07182939350605011, 0.040322981774806976, -0.03299242630600929, -0.035798400640487..."


## 4. Reducción de dimensionalidad con UMAP
UMAP comprime los embeddings para que HDBSCAN trabaje en un espacio más denso. Guardamos tanto el modelo como la proyección para reutilizarla.


In [56]:
if Path(EMB_UMAP_PATH).exists() and Path(UMAP_PATH).exists():
    umap_model = pickle.load(open(UMAP_PATH, "rb"))
    embeddings_umap = np.load(EMB_UMAP_PATH)
    print(f"UMAP cargado desde disco -> {embeddings_umap.shape}")
else:
    umap_model, embeddings_umap = reduce_embeddings(
        embeddings,
        n_neighbors=30,
        n_components=15,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    pickle.dump(umap_model, open(UMAP_PATH, "wb"))
    np.save(EMB_UMAP_PATH, embeddings_umap)
    print(f"UMAP entrenado y guardado -> {embeddings_umap.shape}")

embeddings_umap.shape


UMAP cargado desde disco -> (59458, 15)


(59458, 15)

## 5. Clustering con HDBSCAN
HDBSCAN detecta clusters de densidad variable y marca como `-1` el ruido. Ajusta `min_cluster_size` si quieres más/menos granularidad.


In [57]:
clusterer, cluster_labels = cluster_embeddings( #clustering herarquico
    embeddings_umap,
    min_cluster_size=100, # cantidad de temas (clusters) mínimos
    metric="euclidean",
)

df["cluster_id"] = cluster_labels
pickle.dump(clusterer, open(CLUSTER_MODEL_PATH, "wb"))
df.to_pickle(DF_CLUSTER_PATH)

cluster_counts = df["cluster_id"].value_counts().reset_index()
cluster_counts.columns = ["cluster_id", "size"]
cluster_counts.head()


,cluster_id,size
0,-1,27821
1,19,1537
2,12,985
3,25,936
4,88,909


## 6. Etiquetado legible de cada cluster
Para traducir IDs numéricos a nombres descriptivos:
- Calculamos TF-IDF por cluster (unigrams y bigrams) y extraemos las palabras más representativas.
- Creamos `cluster_label` con esos términos; para `-1` indicamos que es ruido/mixto.
- Generamos un catálogo con tamaño y un ejemplo de review por cluster. Así ves rápidamente qué significa cada número.


In [58]:
def build_cluster_labels(df: pd.DataFrame, text_col: str = "review", cluster_col: str = "cluster_id", top_n: int = 5) -> dict:
    vectorizer = TfidfVectorizer(
        max_features=8000,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
    )
    tfidf = vectorizer.fit_transform(df[text_col])
    vocab = np.array(vectorizer.get_feature_names_out())
    labels = df[cluster_col].values
    cluster_map: dict[int, str] = {}
    for cid in np.unique(labels):
        mask = labels == cid
        scores = tfidf[mask].sum(axis=0).A1
        top_idx = scores.argsort()[::-1][:top_n]
        keywords = [vocab[i] for i in top_idx if scores[i] > 0][:top_n]
        label = ", ".join(keywords) if keywords else "general"
        if cid == -1:
            label = f"ruido/mixed — {label}"
        cluster_map[int(cid)] = label
    return cluster_map


def build_cluster_catalog(df: pd.DataFrame, cluster_label_col: str = "cluster_label") -> pd.DataFrame:
    return (
        df.groupby(["cluster_id", cluster_label_col])
        .agg(
            size=("review", "count"),
            example_review=("review", "first"),
        )
        .reset_index()
        .sort_values("size", ascending=False)
    )


cluster_name_map = build_cluster_labels(df)
df["cluster_label"] = df["cluster_id"].map(cluster_name_map)
cluster_catalog = build_cluster_catalog(df)
cluster_catalog.head(20)


,cluster_id,cluster_label,size,example_review
0,-1,"ruido/mixed — bit, comfortable, color, great, perfect",27821,"It's an okay top, the straps are a bit thin but it fits well and is comfortable enough for everyday wear."
20,19,"skirt, waist, bit, high, high waist",1537,"I was excited to try this skirt, but it didn't quite hit the mark for me - it's a bit too short and the stretchy material isn't as flattering as I'd hoped."
13,12,"tank, comfortable, bit, great, tank great",985,"I've been looking for a simple white tank top like this, and I'm so happy with the purchase, it's great value for the price."
26,25,"hoodie, hood, cozy, soft, bit",936,"Unfortunately, this hoodie is a bit of a letdown - the material feels cheap and the zipper is already breaking after just a few wears."
89,88,"dress, bit, material, black, pattern",909,"I was hoping for a more substantial dress, but this one feels really lightweight and cheap."
36,35,"bikini bottoms, bottoms, bikini, waisted bikini, high",770,"These bikini bottoms are great value for the price, the laser-cut edges are a nice feature, but I wish they came in more colors."
60,59,"vest, comfortable, perfect, straps, bit",761,"I've been looking for a basic black vest top like this for ages, it's great for everyday wear."
2,1,"cardigan, soft, knit, cardigan soft, cozy",758,"The Nora Cardigan is my new favorite piece, the greenish khaki color is so on trend and the sweatshirt fabric is super cozy."
91,90,"material, bit, fit, great, price",750,"I like this top, it's simple and comfy, the black color is perfect for layering or wearing on its own, just wish it came in more sizes."
4,3,"blazer, mariette, mariette blazer, bit, great",744,"The Manson SF slim fit blazer is a great addition to my wardrobe, although the fit is a bit tricky, it's worth the effort."


### Catálogo completo (id -> etiqueta -> tamaño -> ejemplo)
Ejecuta la siguiente celda para ver todas las categorías y qué significa cada número. Ajusta `head()` si quieres limitar.


In [59]:
pd.set_option("display.max_rows", None)
# cluster_catalog
unique_types = cluster_catalog["cluster_id"].dropna().unique()
len(unique_types), unique_types


(92,
 array([-1, 19, 12, 25, 88, 35, 59,  1, 90,  3, 62, 67, 72, 55,  9, 70, 26,
        31, 85, 30, 63, 45, 71,  2, 54,  7,  8, 51, 61,  6, 47,  4, 76, 74,
        43, 22, 44, 34, 28, 10, 42, 21,  5, 79,  0, 23, 17, 84, 64, 32, 37,
        82, 80, 87, 16, 41, 50, 40, 73, 69, 53, 39, 57, 66, 27, 15, 86, 65,
        38, 89, 68, 58, 13, 75, 83, 33, 29, 14, 49, 77, 78, 52, 20, 46, 81,
        48, 56, 24, 18, 11, 36, 60], dtype=int64))

## 7. Persistencia de artefactos
Guardamos embeddings, proyección UMAP, modelo HDBSCAN y DataFrame enriquecido para reutilizar sin recalcular.


In [60]:
np.save(EMB_PATH, embeddings)
np.save(EMB_UMAP_PATH, embeddings_umap)
pickle.dump(umap_model, open(UMAP_PATH, "wb"))
pickle.dump(clusterer, open(CLUSTER_MODEL_PATH, "wb"))
df.to_pickle(DF_CLUSTER_PATH)


## 8. Índice FAISS para búsqueda semántica
Normalizamos los embeddings y creamos un índice `IndexFlatIP` (similaridad coseno). Se guarda en disco para consultas rápidas.


In [61]:
def build_faiss_index(x: np.ndarray) -> faiss.IndexFlatIP:
    index = faiss.IndexFlatIP(x.shape[1])
    faiss.normalize_L2(x)
    index.add(x)
    return index

index = build_faiss_index(embeddings)
faiss.write_index(index, FAISS_INDEX_PATH)


## 9. Búsqueda de reviews similares
Función de consulta que devuelve las reviews más cercanas junto con su `cluster_id` y `cluster_label` para interpretar el resultado.


In [67]:
def search_reviews(user_text: str, top_k: int = 10, df_source: pd.DataFrame = df, model_ref: SentenceTransformer = model, index_ref = index):
    query_emb = model_ref.encode([user_text])
    faiss.normalize_L2(query_emb)
    scores, idxs = index_ref.search(query_emb, k=top_k)
    return df_source.iloc[idxs[0]][["product_code", "review","review_stars", "cluster_id", "cluster_label"]]

user_text = "I’m looking for a scary unicorn."
results = search_reviews(user_text, top_k=10)
results


,product_code,review,review_stars,cluster_id,cluster_label
16043,628958,"My kid's Unicorn onesie is a total hit, it's so soft and the embroidered details are adorable, we've worn it to a few costume parties and it's been a real showstopper.",5,39,"adorable, baby, set, organic cotton, organic"
19831,661413,"I was really disappointed with the Halloween A-band unicorn, it looks cheap and the satin covering is already coming off after one use.",1,32,"headband, aliceband, knot, accessory, touch"
28864,705095,"This adorable unicorn horn headband is perfect for my little girl's dress-up adventures, the soft faux fur ears and tulle decoration make it a delightful accessory.",5,32,"headband, aliceband, knot, accessory, touch"
52020,830346,"My daughter loves this unicorn horn headband, it's so sparkly and cute, perfect for dress-up.",5,32,"headband, aliceband, knot, accessory, touch"
52749,835136,"The Scary spice cheeky highwaist in dark green is a decent choice, but it's not the most comfortable fit and the material could be better.",3,-1,"ruido/mixed — bit, comfortable, color, great, perfect"
52748,835136,"The dark green Scary spice cheeky highwaist is okay, but the color faded quickly and the fit wasn't quite right.",2,-1,"ruido/mixed — bit, comfortable, color, great, perfect"
16626,633675,"My daughter loves her new unicorn headband, the glittery plastic and fabric ears are so cute and it's well-made too.",5,32,"headband, aliceband, knot, accessory, touch"
52747,835136,"Don't waste your money on the Scary spice cheeky highwaist, it's just not worth it, the quality is subpar.",1,-1,"ruido/mixed — bit, comfortable, color, great, perfect"
7310,552826,"My black Beetle beanie is okay, but it's not particularly remarkable - it's a solid, simple design that works, but doesn't wow.",3,14,"beanie, soft, beanie soft, winter, knit"
1884,368038,"The Biq Square Claw in black is a decent hair accessory, but it's not particularly strong and tends to slip out of my hair if it's styled a certain way.",3,13,"hair, clips, hair clips, hair ties, ties"


## SIGUIENTES SON SOLO PROPUESTAS

## 10. Búsqueda two-stage por clusters + fallback a ruido
Estrategia para acelerar: primero se escogen los clusters más cercanos vía centroides, luego se busca solo en sus reviews. Si no alcanza `top_k`, se completa con el cluster de ruido (`-1`).

Flujo:
1) Precomputar centroides por `cluster_id` (excluyendo `-1`) y su índice FAISS pequeño.
2) Mantener un índice FAISS del ruido para fallback.
3) Consulta: seleccionar N clusters más cercanos, buscar en ese subconjunto, y si falta, completar con ruido.


In [63]:
# Precomputar estructuras para búsqueda two-stage (centroides + ruido)
valid_df = df[df.cluster_id != -1].reset_index()
embs = embeddings  # alias corto

cluster_to_idx = (
    valid_df.groupby("cluster_id")["index"]
    .apply(list)
    .to_dict()
)

cluster_centroids = {
    cid: embs[idxs].mean(axis=0).astype("float32")
    for cid, idxs in cluster_to_idx.items()
}
centroid_ids = list(cluster_centroids.keys())
centroid_matrix = (
    np.vstack(list(cluster_centroids.values())).astype("float32")
    if centroid_ids else np.zeros((0, embs.shape[1]), dtype="float32")
)
faiss.normalize_L2(centroid_matrix)
centroid_index = faiss.IndexFlatIP(centroid_matrix.shape[1])
if len(centroid_ids) > 0:
    centroid_index.add(centroid_matrix)

# Índice de ruido (-1) para fallback
noise_idx = df.index[df.cluster_id == -1].to_numpy()
if len(noise_idx) > 0:
    embs_noise = embs[noise_idx].astype("float32", copy=True)
    faiss.normalize_L2(embs_noise)
    noise_index = faiss.IndexFlatIP(embs_noise.shape[1])
    noise_index.add(embs_noise)
else:
    noise_index = None


In [68]:
def search_two_stage_with_noise(
    user_text: str,
    top_clusters: int = 3,
    top_k: int = 10,
) -> pd.DataFrame:
    # Busca primero en los clusters más cercanos (por centroides) y luego en ruido si falta.
    if len(centroid_ids) == 0:
        return search_reviews(user_text, top_k=top_k)

    query_emb = model.encode([user_text])
    faiss.normalize_L2(query_emb)

    # 1) Elegir clusters más cercanos
    _, c_idxs = centroid_index.search(query_emb, k=min(top_clusters, len(centroid_ids)))
    candidate_rows: list[int] = []
    for cid in [centroid_ids[i] for i in c_idxs[0]]:
        candidate_rows.extend(cluster_to_idx[cid])

    if not candidate_rows:
        return search_reviews(user_text, top_k=top_k)

    # 2) Búsqueda en subconjunto de clusters seleccionados
    subset = np.array(candidate_rows, dtype=int)
    sub_embs = embs[subset].astype("float32", copy=True)
    faiss.normalize_L2(sub_embs)
    sub_index = faiss.IndexFlatIP(sub_embs.shape[1])
    sub_index.add(sub_embs)
    _, s_idxs = sub_index.search(query_emb, k=min(top_k, len(subset)))
    hits = subset[s_idxs[0]]

    # 3) Fallback al ruido si faltan resultados
    if len(hits) < top_k and noise_index is not None and len(noise_idx) > 0:
        need = top_k - len(hits)
        _, n_idxs = noise_index.search(query_emb, k=min(need, len(noise_idx)))
        hits = np.concatenate([hits, noise_idx[n_idxs[0]]])

    return df.iloc[hits][["product_code", "review", "review_stars", "cluster_id", "cluster_label"]]


fast_results = search_two_stage_with_noise(
    "I’m looking for a cute unicorn.",
    top_clusters=3,
    top_k=10,
)
fast_results


,product_code,review,review_stars,cluster_id,cluster_label
44004,776532,"My baby loves this adorable beanie - the ear flaps and ties under the chin are so sweet, and the light grey color is perfect for our little one.",4,14,"beanie, soft, beanie soft, winter, knit"
7310,552826,"My black Beetle beanie is okay, but it's not particularly remarkable - it's a solid, simple design that works, but doesn't wow.",3,14,"beanie, soft, beanie soft, winter, knit"
8035,556539,"This light pink scrunchie is adorable and works perfectly for my daily ponytail, love the velvet texture.",5,15,"scrunchie, hair, velvet, velvet scrunchie, scrunchies"
8002,556539,"This scrunchie is not only cute, but it's also ridiculously soft and gentle on my hair; great value for the price.",5,15,"scrunchie, hair, velvet, velvet scrunchie, scrunchies"
8013,556539,"I'm so happy with this beige scrunchie, it's exactly what I was looking for, and the price is a steal.",5,15,"scrunchie, hair, velvet, velvet scrunchie, scrunchies"
23429,682261,"I'm really happy with my SALLY BEANIE, the yellow color is bright and cheerful, and the sewn-in turn-up at the hem is a nice touch.",4,14,"beanie, soft, beanie soft, winter, knit"
44199,777815,I'm a big fan of the Flirty Vinnie ancle in turquoise - the plastic beads add a fun pop of colour and the adjustable length is really convenient.,5,5,"necklace, adjustable length, chain, gold, flirty"
7312,552826,"This beanie is okay, the striped pattern is nice but the beige color is a bit dull, and the appliqué on the front is a cute touch.",3,14,"beanie, soft, beanie soft, winter, knit"
7196,550412,"Unfortunately, this pink beanie fell apart after a few wearings, and the faux fur pompom came loose.",2,14,"beanie, soft, beanie soft, winter, knit"
7308,552826,"Unfortunately, my Beetle beanie didn't quite live up to expectations - it started to lose its shape after a few wearings, and the appliqué came loose.",2,14,"beanie, soft, beanie soft, winter, knit"


## 11. Búsqueda rápida: cluster único
Variante más veloz: se elige el cluster más cercano por centroides y se busca solo dentro de ese cluster con FAISS. Mayor rapidez, menor cobertura si la query cae entre temas.


In [69]:
def search_cluster_first(user_text: str, top_k: int = 10) -> pd.DataFrame:
    # Elige el cluster más parecido (por centroides) y busca solo dentro de él.
    if len(centroid_ids) == 0:
        return search_reviews(user_text, top_k=top_k)

    query_emb = model.encode([user_text])
    faiss.normalize_L2(query_emb)

    # 1) Cluster más cercano
    _, c_idxs = centroid_index.search(query_emb, k=1)
    best_cid = centroid_ids[int(c_idxs[0][0])]
    candidate_rows = cluster_to_idx.get(best_cid, [])
    if not candidate_rows:
        return search_reviews(user_text, top_k=top_k)

    # 2) Búsqueda dentro de ese cluster
    subset = np.array(candidate_rows, dtype=int)
    sub_embs = embs[subset].astype("float32", copy=True)
    faiss.normalize_L2(sub_embs)
    sub_index = faiss.IndexFlatIP(sub_embs.shape[1])
    sub_index.add(sub_embs)
    _, s_idxs = sub_index.search(query_emb, k=min(top_k, len(subset)))
    hits = subset[s_idxs[0]]

    return df.iloc[hits][["product_code", "review", "review_stars", "cluster_id", "cluster_label"]]

# Ejemplo de uso
fast_cluster_results = search_cluster_first(
    "I’m looking for a cute unicorn.",
    top_k=10,
)
fast_cluster_results


,product_code,review,review_stars,cluster_id,cluster_label
44004,776532,"My baby loves this adorable beanie - the ear flaps and ties under the chin are so sweet, and the light grey color is perfect for our little one.",4,14,"beanie, soft, beanie soft, winter, knit"
7310,552826,"My black Beetle beanie is okay, but it's not particularly remarkable - it's a solid, simple design that works, but doesn't wow.",3,14,"beanie, soft, beanie soft, winter, knit"
23429,682261,"I'm really happy with my SALLY BEANIE, the yellow color is bright and cheerful, and the sewn-in turn-up at the hem is a nice touch.",4,14,"beanie, soft, beanie soft, winter, knit"
7312,552826,"This beanie is okay, the striped pattern is nice but the beige color is a bit dull, and the appliqué on the front is a cute touch.",3,14,"beanie, soft, beanie soft, winter, knit"
7196,550412,"Unfortunately, this pink beanie fell apart after a few wearings, and the faux fur pompom came loose.",2,14,"beanie, soft, beanie soft, winter, knit"
7308,552826,"Unfortunately, my Beetle beanie didn't quite live up to expectations - it started to lose its shape after a few wearings, and the appliqué came loose.",2,14,"beanie, soft, beanie soft, winter, knit"
18320,649555,"The black beanie is okay, but the faux fur pompom feels a bit cheap and sheds a lot.",2,14,"beanie, soft, beanie soft, winter, knit"
35044,732409,"This Beetle beanie is so soft and cozy, I love the grey color and the way it stays on my head even on windy days.",5,14,"beanie, soft, beanie soft, winter, knit"
5611,517046,"The appliqué on this beanie is cute, but the fabric feels thin and the fit is a bit off.",2,14,"beanie, soft, beanie soft, winter, knit"
40163,756099,"I adore the Pinnochio Beanie in Light Beige, it's incredibly soft and the sewn-in turn-up at the hem is a thoughtful detail, just what I needed for the cold months.",5,14,"beanie, soft, beanie soft, winter, knit"
